# 02b - Auto-Annotation with Grounding DINO
## Bradford Bulls - AI Sponsorship Exposure Valuation System

**Problem:** 6000+ frames is too many to annotate manually.

**Solution:** 
1. Select ~500-800 most diverse frames (skip near-duplicates)
2. Use **Grounding DINO** (zero-shot object detection) to auto-detect logos by text prompt
3. Generate YOLO-format annotation files
4. Upload images + annotations to Roboflow → only need to **review & correct**

This reduces manual work from drawing 6000+ bounding boxes to just reviewing/fixing pre-drawn ones.

## 1. Install Dependencies

In [1]:
# !pip install autodistill autodistill-grounding-dino supervision -q

In [2]:
import sys
sys.path.insert(0, "..")

import cv2
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from tqdm import tqdm

from src.config import (
    FRAMES_CLEAR_DIR, METADATA_DIR, OUTPUT_DIR,
    SPONSOR_LABELS, DEVICE
)

print(f"Device: {DEVICE}")
print(f"Frames available: {len(list(FRAMES_CLEAR_DIR.glob('frame_*.jpg')))}")

[Device] Using: cuda (NVIDIA GeForce RTX 5060 Ti)
Device: cuda
Frames available: 6476


## 2. Select Diverse Frame Subset

From 6000+ frames, select ~500-800 most diverse frames using visual similarity filtering.
This avoids annotating near-identical frames and gives better training data diversity.

In [3]:
import imagehash
from PIL import Image

# ============================================================
# CONFIG: How many frames to select for annotation
# ============================================================
TARGET_ANNOTATION_COUNT = 600  # Adjust as needed (500-800 recommended)
DIVERSITY_HASH_THRESHOLD = 8  # Higher = more similar frames allowed

all_frames = sorted(FRAMES_CLEAR_DIR.glob("frame_*.jpg"))
print(f"Total clear frames: {len(all_frames)}")

# Strategy: Walk through frames, keep only those that are visually different
# from the last kept frame (using perceptual hash)
selected_frames = []
last_hash = None

for frame_path in tqdm(all_frames, desc="Selecting diverse frames"):
    img = Image.open(frame_path)
    current_hash = imagehash.phash(img)
    
    if last_hash is None or (current_hash - last_hash) >= DIVERSITY_HASH_THRESHOLD:
        selected_frames.append(frame_path)
        last_hash = current_hash

print(f"\nAfter diversity filtering: {len(selected_frames)} frames")

# If still too many, subsample evenly
if len(selected_frames) > TARGET_ANNOTATION_COUNT:
    step = len(selected_frames) / TARGET_ANNOTATION_COUNT
    indices = [int(i * step) for i in range(TARGET_ANNOTATION_COUNT)]
    selected_frames = [selected_frames[i] for i in indices]
    print(f"After subsampling: {len(selected_frames)} frames")

print(f"\nSelected {len(selected_frames)} frames for auto-annotation")

Total clear frames: 6476


Selecting diverse frames: 100%|██████████| 6476/6476 [00:33<00:00, 191.48it/s]


After diversity filtering: 6158 frames
After subsampling: 600 frames

Selected 600 frames for auto-annotation


In [4]:
# Copy selected frames to a separate directory for annotation
ANNOTATION_DIR = OUTPUT_DIR / "frames_for_annotation"
ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

for fp in tqdm(selected_frames, desc="Copying selected frames"):
    shutil.copy2(fp, ANNOTATION_DIR / fp.name)

print(f"Copied {len(selected_frames)} frames to: {ANNOTATION_DIR}")

Copying selected frames: 100%|██████████| 600/600 [00:00<00:00, 1765.61it/s]

Copied 600 frames to: c:\Users\Admin\Desktop\bradford_bulls\notebooks\..\output\frames_for_annotation


## 3. Auto-Annotate with Grounding DINO

Grounding DINO is a **zero-shot** object detector — it can find objects by text description without any training.

We use text prompts like `"AON logo"`, `"sponsor logo on jersey"`, `"text on shirt"` to detect logo regions automatically.

In [5]:
from autodistill_grounding_dino import GroundingDINO
from autodistill.detection import CaptionOntology

# ============================================================
# ONTOLOGY: Map text prompts → label classes
# ============================================================
ontology = CaptionOntology({
    "AON logo": "aon",
    "AON text on jersey": "aon",
    "MCP logo": "mcp",
    "MCP text on jersey": "mcp",
    "Cedar Court Hotels logo": "cch_cedar_court",
    "ChadLaw logo": "chadlaw",
    "ATM Hospitality logo": "atm_hospitality",
    "EM workwear logo": "em_workwear",
    "Fairway Flooring logo": "fairway_flooring",
    "KLG logo": "klg",
    "MNA logo": "mna_cladding",
    "MNA text on shorts": "mna_support",
    "Bartercard logo": "bartercard",
    "Top Notch logo": "top_notch",
    "Romantica logo": "romantica_beds",
    "sponsor logo on rugby jersey": "sponsor_logo",
    "text on rugby shirt": "sponsor_logo",
    "brand logo on sportswear": "sponsor_logo",
})

print("Ontology configured with", len(ontology.promptMap), "prompts")
print("\nPrompt → Label mappings:")
for prompt, label in ontology.promptMap:
    print(f"  '{prompt}' → {label}")

Ontology configured with 18 prompts

Prompt → Label mappings:
  'AON logo' → aon
  'AON text on jersey' → aon
  'MCP logo' → mcp
  'MCP text on jersey' → mcp
  'Cedar Court Hotels logo' → cch_cedar_court
  'ChadLaw logo' → chadlaw
  'ATM Hospitality logo' → atm_hospitality
  'EM workwear logo' → em_workwear
  'Fairway Flooring logo' → fairway_flooring
  'KLG logo' → klg
  'MNA logo' → mna_cladding
  'MNA text on shorts' → mna_support
  'Bartercard logo' → bartercard
  'Top Notch logo' → top_notch
  'Romantica logo' → romantica_beds
  'sponsor logo on rugby jersey' → sponsor_logo
  'text on rugby shirt' → sponsor_logo
  'brand logo on sportswear' → sponsor_logo


Importing from timm.models.layers is deprecated, please import via timm.layers


In [6]:
# ============================================================
# RUN AUTO-ANNOTATION
# ============================================================
AUTOLABEL_OUTPUT = OUTPUT_DIR / "auto_annotated"
AUTOLABEL_OUTPUT.mkdir(parents=True, exist_ok=True)

base_model = GroundingDINO(
    ontology=ontology,
    box_threshold=0.2,
    text_threshold=0.15,
)

base_model.label(
    input_folder=str(ANNOTATION_DIR),
    output_folder=str(AUTOLABEL_OUTPUT),
)

print(f"\nAuto-annotation complete!")

torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4383.)


trying to load grounding dino directly
final text_encoder_type: bert-base-uncased


`resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
Labeling c:\Users\Admin\Desktop\bradford_bulls\notebooks\..\output\frames_for_annotation\frame_000028_00m18s.jpg:   0%|          | 0/600 [00:00<?, ?it/s]The `device` argument is deprecated and will be removed in v5 of Transformers.
Requested unified CUBLASLT workspace size of 1048576 bytes exceeds CUBLAS workspace size of 131072 bytes. Please increase CUBLAS workspace size via CUBLAS_WORKSPACE_CONFIG or decrease requested CUBLASLT_WORKSPACE_SIZE. Otherwise CUBLASLT workspace size will be limited to the CUBLAS workspace size. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\cuda\CublasHandlePool.cpp:233.)
torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_r

Labeled dataset created - ready for distillation.

Auto-annotation complete!


## 4. Preview Auto-Annotations

In [7]:
class_names = list(dict.fromkeys([label for prompt, label in ontology.promptMap]))
print(f"Class names: {class_names}\n")

def visualize_annotation(image_path, label_path, class_names):
    img = cv2.imread(str(image_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    fig, ax = plt.subplots(1, 1, figsize=(12, 7))
    ax.imshow(img_rgb)
    if label_path.exists():
        with open(label_path) as f: lines = f.readlines()
        colors = plt.cm.Set3(np.linspace(0, 1, len(class_names)))
        for line in lines:
            parts = line.strip().split()
            cls_id = int(parts[0])
            cx, cy, bw, bh = [float(x) for x in parts[1:5]]
            x1 = (cx - bw/2) * w
            y1 = (cy - bh/2) * h
            label = class_names[cls_id] if cls_id < len(class_names) else f"class_{cls_id}"
            color = colors[cls_id % len(colors)]
            rect = patches.Rectangle((x1, y1), bw*w, bh*h, linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1-5, label, color=color, fontweight='bold', backgroundcolor='black')
    ax.axis("off")
    plt.show()

Class names: ['aon', 'mcp', 'cch_cedar_court', 'chadlaw', 'atm_hospitality', 'em_workwear', 'fairway_flooring', 'klg', 'mna_cladding', 'mna_support', 'bartercard', 'top_notch', 'romantica_beds', 'sponsor_logo']

